# Leaderboard Validation — All Classifiers

Pipeline finale locked. Il validation seleziona; il test valuta. Il dry-run non carica modelli e non scrive artefatti scientifici.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "configs/final_classifier_registry.json").is_file():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
from final_classifier_evaluation import *

DRY_RUN = True
RECOMPUTE_TEST_PREDICTIONS = False
ALLOW_UNVERIFIED_LEGACY_PREDICTIONS = False
TEST_BATCH_SIZE = 8
TEST_NUM_WORKERS = 4
DEVICE = "auto"
PATIENT_AGGREGATION = "mean"
TEST_CSV = PROJECT_ROOT / "data/processed/metadata/test.csv"
TEST_DATASET_MANIFEST = PROJECT_ROOT / "results/final_evaluation/test_dataset_manifest.json"
REGISTRY_PATH = PROJECT_ROOT / "configs/final_classifier_registry.json"

DRY_RUN = True
PRIMARY_SELECTION_METRIC = "validation_roc_auc"
SECONDARY_SELECTION_METRIC = "validation_pr_auc"
FINALIST_POLICY = {"include_baseline_per_architecture": True, "include_best_synthetic_per_architecture": True, "include_best_augmented_per_architecture": True, "include_best_overall_per_architecture": True, "max_finalists_total": 10}
OUTPUT_DIR = PROJECT_ROOT / "results/final_evaluation"

In [ ]:
registry = build_experiment_registry(REGISTRY_PATH)
coverage = build_test_coverage_table(registry, PROJECT_ROOT)
finalists = select_validation_finalists(coverage, FINALIST_POLICY)
print(coverage[["experiment_id", "architecture", "validation_roc_auc", "scientifically_eligible", "selected_by_validation", "status", "blocked_reason", "exclusion_reason"]].to_string(index=False))
print("Finalisti validation:", [(x["experiment_id"], x["selection_reason"]) for x in finalists])
assert not any(key.startswith("test_") and key.endswith(("auc", "accuracy", "f1")) for key in [PRIMARY_SELECTION_METRIC, SECONDARY_SELECTION_METRIC])
if DRY_RUN: print("DRY_RUN: leaderboard/manifest/lock non scritti e metriche test non lette.")

In [ ]:
if not DRY_RUN:
    import matplotlib.pyplot as plt
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True); (OUTPUT_DIR / "figures").mkdir(exist_ok=True)
    leaderboard = coverage.rename(columns={"training_dataset_variant": "dataset_variant"})
    leaderboard.to_csv(OUTPUT_DIR / "validation_leaderboard.csv", index=False)
    (OUTPUT_DIR / "validation_leaderboard.json").write_text(strict_json_dumps(leaderboard.to_dict("records"), indent=2) + "\n")
    # Scientific finalists are kept even without a checkpoint (e.g. ResNet): build_locked_finalist_entries
    # never raises on a missing checkpoint and preserves the full scientific/operational field set.
    locked_entries = build_locked_finalist_entries(finalists, PROJECT_ROOT)
    manifest = lock_finalists_manifest(locked_entries, FINALIST_POLICY, OUTPUT_DIR / "finalists_manifest.json")
    (OUTPUT_DIR / "FINALISTS_LOCKED").write_text(manifest["lock_signature"]["sha256"] + "\n")
    plot = leaderboard[leaderboard.validation_roc_auc.notna()].sort_values("validation_roc_auc")
    ax = plot.plot.barh(x="display_name", y="validation_roc_auc", legend=False, figsize=(9, 8)); ax.figure.tight_layout(); ax.figure.savefig(OUTPUT_DIR / "figures/validation_auc_comparison.png", dpi=300); plt.close(ax.figure)
    pr = leaderboard[leaderboard.validation_pr_auc.notna()].sort_values("validation_pr_auc")
    ax = pr.plot.barh(x="display_name", y="validation_pr_auc", legend=False, figsize=(9, 6)); ax.figure.tight_layout(); ax.figure.savefig(OUTPUT_DIR / "figures/validation_pr_auc_comparison.png", dpi=300); plt.close(ax.figure)